# Training NER (task 1) — Colab

Melatih baseline **BiLSTM-CRF** dan **IndoBERT-base-p1** di NERGrit (label PER, ORG, LOC), masing-masing 3 seed.
Semua logika ada di `src/indonesian_hoax_checker/ner/`; notebook ini hanya menjalankannya.

Cara pakai:
1. `Runtime > Change runtime type > GPU`.
2. Pastikan kode sudah di-push ke branch `BRANCH` di GitHub.
3. Jalankan semua sel dari atas. Hasil di-upload ke folder Drive tim `hasil_training`
   (akun Google yang dipakai di Colab harus punya akses edit ke folder itu).

In [ ]:
# Pengaturan
REPO_URL = "https://github.com/danenftyessir/indonesian-hoax-checker.git"
BRANCH = "feat/ner"
SEEDS = "13 42 2026"

# Folder Drive tim: tubes_kelompok9 > hasil_training
# https://drive.google.com/drive/folders/1Eq9C3SIW42xYpRn5UVAwoiLrRkOY52Pn
DRIVE_FOLDER_ID = "1Eq9C3SIW42xYpRn5UVAwoiLrRkOY52Pn"

# True = ambil checkpoint dan metrik tahap 1 dari run sebelumnya di Drive, jadi baseline tidak dilatih ulang.
# RESTORE_RUN = nama subfolder (mis. "ner_20261111-2010"); kosong = run terbaru yang punya ner_results.zip.
RESTORE_FROM_DRIVE = True
RESTORE_RUN = "ner_20261111-1312"  # run baseline tahap 1 (BiLSTM-CRF dan IndoBERT di NERGrit); kosong = run terbaru

## 1. Cek GPU dan login Google (untuk upload hasil)

In [ ]:
!nvidia-smi

from google.colab import auth
auth.authenticate_user()

## 2. Ambil kode dan install dependensi

Kalau repo private, simpan token GitHub di Colab Secrets dengan nama `GITHUB_TOKEN`.

In [ ]:
import os

# Pakai token kalau repo private
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    url = REPO_URL

if not os.path.exists("/content/repo"):
    !git clone -q -b {BRANCH} {url} /content/repo
%cd /content/repo
!git pull -q
!git log --oneline -1
!pip install -q -e ".[training]"

## 3. Siapkan data NERGrit dan NERP

In [ ]:
!python -m indonesian_hoax_checker.ner.prepare

## 3b. Pulihkan hasil tahap 1 dari Drive (opsional)

Mengunduh `ner_results.zip` dari subfolder `ner_*` di `hasil_training` lalu mengekstraknya ke repo
(checkpoint, metrik, prediksi). Sel 4 dan 5 otomatis dilewati bila ringkasannya sudah ada.

In [ ]:
import io
import zipfile

from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload


def find_results_zip(drive, parent_id, run_name=""):
    # Subfolder ner_* terbaru dulu (rq1_ner_* tidak ikut)
    query = f"'{parent_id}' in parents and mimeType = 'application/vnd.google-apps.folder' and trashed = false"
    folders = drive.files().list(q=query, orderBy="createdTime desc", fields="files(id, name)",
                                 supportsAllDrives=True, includeItemsFromAllDrives=True).execute()["files"]
    for folder in folders:
        if not folder["name"].startswith("ner_") or (run_name and folder["name"] != run_name):
            continue
        query = f"'{folder['id']}' in parents and name = 'ner_results.zip' and trashed = false"
        files = drive.files().list(q=query, fields="files(id)", supportsAllDrives=True,
                                   includeItemsFromAllDrives=True).execute()["files"]
        if files:
            return folder["name"], files[0]["id"]
    return None, None


if RESTORE_FROM_DRIVE:
    drive = build("drive", "v3")
    run_name, file_id = find_results_zip(drive, DRIVE_FOLDER_ID, RESTORE_RUN)
    if file_id is None:
        print("Tidak ada ner_results.zip di Drive; tahap 1 akan dilatih dari awal.")
    else:
        # Unduh ke file, bukan memori: zip berisi checkpoint IndoBERT (~1,5 GB)
        with io.FileIO("/content/restore.zip", "wb") as f:
            downloader = MediaIoBaseDownload(f, drive.files().get_media(fileId=file_id, supportsAllDrives=True))
            done = False
            while not done:
                _, done = downloader.next_chunk()
        zipfile.ZipFile("/content/restore.zip").extractall(".")
        print(f"Dipulihkan dari {run_name}")
        !ls artifacts/checkpoints/ner outputs/metrics

## 4. Latih BiLSTM-CRF (baseline)

In [ ]:
# Dilewati bila hasilnya sudah dipulihkan dari Drive
if os.path.exists("outputs/metrics/ner_bilstm_crf_summary.json"):
    print("ner_bilstm_crf sudah ada, dilewati")
else:
    !python -m indonesian_hoax_checker.ner.train --config configs/models/ner_bilstm_crf.yaml --seeds {SEEDS} 2>&1 | grep -v Warning

## 5. Latih IndoBERT-base-p1

In [ ]:
# Dilewati bila hasilnya sudah dipulihkan dari Drive
if os.path.exists("outputs/metrics/ner_indobert_summary.json"):
    print("ner_indobert sudah ada, dilewati")
else:
    !python -m indonesian_hoax_checker.ner.train --config configs/models/ner_indobert.yaml --seeds {SEEDS} 2>&1 | grep -v Warning

## 5b. Tahap 2: CLAIM dan TIME (setelah seed set dianotasi)

Melanjutkan model tahap 1 (NERGrit) dengan seed set yang punya kelima label.
Butuh `data/annotations/ner/seed_adjudicated.jsonl` di repo (hasil `ner.annotation build`).
Kalau file itu belum ada, sel ini dilewati.

In [ ]:
if os.path.exists("data/annotations/ner/seed_adjudicated.jsonl"):
    !python -m indonesian_hoax_checker.ner.annotation split
    !python -m indonesian_hoax_checker.ner.train --config configs/models/ner_bilstm_crf_claim_time.yaml --seeds {SEEDS} 2>&1 | grep -v Warning
    !python -m indonesian_hoax_checker.ner.train --config configs/models/ner_indobert_claim_time.yaml --seeds {SEEDS} 2>&1 | grep -v Warning
else:
    print("Seed set belum ada, tahap 2 dilewati.")

## 5c. Tahap 2 dengan layer terpisah (entitas dan CLAIM)

Pada model gabungan (5b), label CLAIM menimpa PER/ORG/LOC/TIME di dalam span klaim (sekitar separuh entitas seed set), jadi model belajar bahwa isi klaim tidak berisi entitas. Di sini entitas dan CLAIM dilatih sebagai dua model dari model NERGrit yang sama:

- `ner_indobert_entity`: PER, ORG, LOC, TIME (entitas di dalam klaim tetap berlabel)
- `ner_indobert_claim`: hanya span CLAIM

Bandingkan dengan `ner_indobert_claim_time` (gabungan) pada F1 CLAIM, F1 entitas, `nergrit_test`, dan `seed_informal_test`.

In [ ]:
if os.path.exists("data/processed/ner/seed_entity_train.conll"):
    !python -m indonesian_hoax_checker.ner.train --config configs/models/ner_indobert_entity.yaml --seeds {SEEDS} 2>&1 | grep -v Warning
    !python -m indonesian_hoax_checker.ner.train --config configs/models/ner_indobert_claim.yaml --seeds {SEEDS} 2>&1 | grep -v Warning
else:
    print("seed_entity_*.conll belum ada; jalankan sel 5b (annotation split) dulu.")

## 6. Ringkasan hasil (F1 test, rata-rata ± simpangan baku 3 seed)

In [ ]:
import json
import pandas as pd

rows = []
for name in ["ner_bilstm_crf", "ner_indobert", "ner_bilstm_crf_claim_time", "ner_indobert_claim_time", "ner_indobert_entity", "ner_indobert_claim"]:
    path = f"outputs/metrics/{name}_summary.json"
    if not os.path.exists(path):
        continue
    s = json.load(open(path))
    row = {"model": name, "F1": f"{s['test_f1']['mean']:.4f} ± {s['test_f1']['std']:.4f}"}
    for kind, v in s["test_f1_per_type"].items():
        row[kind] = f"{v['mean']:.4f} ± {v['std']:.4f}"
    for key, v in s.get("extra_f1", {}).items():
        row[key] = f"{v['mean']:.4f} ± {v['std']:.4f}"
    rows.append(row)
pd.DataFrame(rows)

## Error analysis (seed 42)

Kesalahan per tipe: salah tipe, salah batas, terlewat, dan salah deteksi, beserta contoh kalimat. Hasil JSON ikut di-upload.

In [ ]:
!python -m indonesian_hoax_checker.ner.error_analysis outputs/predictions/ner_bilstm_crf_seed42.conll outputs/predictions/ner_indobert_seed42.conll --examples 3

## Coba prediksi dari teks mentah

Memakai `ner.predict` dengan checkpoint seed 42 tiap model.

In [ ]:
import sys

sys.path.insert(0, "/content/repo/src")  # paket hasil pip install -e belum terbaca kernel ini
from indonesian_hoax_checker.ner.predict import NERPredictor

contoh = [
    "Presiden Joko Widodo meresmikan Bandara Dhoho di Kediri, Jawa Timur.",
    "Beredar pesan bahwa Kementerian Kesehatan membagikan vaksin gratis di Surabaya.",
    "pemerintah stop bansos bsk gaes!!!",
]
for name in ["ner_bilstm_crf", "ner_indobert"]:
    predictor = NERPredictor(f"artifacts/checkpoints/ner/{name}_seed42")
    print(f"== {name}")
    for text, result in zip(contoh, predictor.predict_batch(contoh)):
        print(text)
        print("  ", [(e["text"], e["label"]) for e in result["entities"]])

## 7. Upload hasil ke Drive tim

Isi `hasil_training/`:
- `ner_<waktu>/` berisi file ringkasan `*_summary.json` (bisa langsung dibuka)
- `ner_<waktu>/ner_results.zip` berisi checkpoint, metrik, dan prediksi semua seed

In [ ]:
from datetime import datetime

from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

drive = build("drive", "v3")


def make_folder(name, parent_id):
    body = {"name": name, "mimeType": "application/vnd.google-apps.folder", "parents": [parent_id]}
    return drive.files().create(body=body, fields="id", supportsAllDrives=True).execute()["id"]


def upload(path, parent_id):
    media = MediaFileUpload(path, resumable=True)
    body = {"name": os.path.basename(path), "parents": [parent_id]}
    drive.files().create(body=body, media_body=media, fields="id", supportsAllDrives=True).execute()
    print("uploaded", path)


# Satu subfolder per run supaya hasil lama tidak tertimpa
run_folder = make_folder(f"ner_{datetime.now():%Y%m%d-%H%M}", DRIVE_FOLDER_ID)

# Dipisah supaya satu kegagalan upload tidak menghilangkan semuanya, dan ukurannya kecil:
#   ner_results.zip     metrik dan prediksi semua run (kecil)
#   ner_checkpoints_seed42.zip   checkpoint seed 42 model tahap 2 saja (baseline tahap 1 sudah ada di Drive)
!rm -f /content/ner_results.zip /content/ner_checkpoints_seed42.zip  # zip -r menambah ke zip lama bila ada
!zip -q -r /content/ner_results.zip outputs/metrics outputs/predictions
!cd artifacts/checkpoints/ner && zip -q -r /content/ner_checkpoints_seed42.zip $(ls -d *claim_time_seed42 *entity_seed42 *claim_seed42 2>/dev/null)
for name in ["ner_bilstm_crf", "ner_indobert", "ner_bilstm_crf_claim_time", "ner_indobert_claim_time", "ner_indobert_entity", "ner_indobert_claim"]:
    path = f"outputs/metrics/{name}_summary.json"
    if os.path.exists(path):
        upload(path, run_folder)

# Upload zip dicoba satu per satu; bila kuota Drive penuh (403 storageQuotaExceeded), file tetap ada di /content
for zip_path in ["/content/ner_results.zip", "/content/ner_checkpoints_seed42.zip"]:
    try:
        upload(zip_path, run_folder)
    except Exception as err:
        print(f"GAGAL upload {zip_path}: {err}\nKuota Drive penuh? Kosongkan Drive, lalu jalankan ulang sel ini (file masih ada di /content).")
print(f"https://drive.google.com/drive/folders/{run_folder}")
